In [8]:
import pandas as pd
import os

from tqdm import tqdm

from pprint import pprint

from rac.zdb import get_zdb_meta_data


In [9]:
!ls ../

article-separation-annotations-2026-05-19-19-19-07.839805-errors.md
article-separation-annotations-2026-05-19-19-19-07.839805.json
as-eval.txt
backup
BnF-images
BnF.zip
errors.md
ey5-ocr-BnF-images
ey5-ocr-NLF-images
ey5-ocr-SBB-images
ey8-ocr-BnF-images
ey8-ocr-NLF-images
ey8-ocr-SBB-images
ey-ocr-BnF-images
ey-ocr-NLF-images
ey-ocr-SBB-images
ey-seg-BnF-images
ey-seg-NLF-images
ey-seg-SBB-images
GT-ArticleSeparation-NewsEye
gt-BnF.tsv
gt-NLF.tsv
gt-SBB.tsv
Makefile
Makefile.backup
match-gt-BnF-eynollah-layout-ocr.tsv
match-gt-BnF-gt-layout-ocr.tsv
match-gt-BnF-pero-layout-ocr.tsv
match-gt-NLF-eynollah-layout-ocr.tsv
match-gt-NLF-gt-layout-ocr.tsv
match-gt-NLF-pero-layout-ocr.tsv
match-gt-SBB-eynollah-layout-ocr.tsv
match-gt-SBB-pero-layout-ocr.tsv
models
NLF-images
NLF.zip
notebook
pero_eu_cz_print_newspapers_2022-09-26
pero_eu_cz_print_newspapers_2022-09-26.zip
pero-ocr-BnF-images
pero-ocr-BnF-images.zip
pero-ocr-NLF-images
pero-ocr-NLF-images.zip
pero-ocr-SBB-images
pero-ocr-SBB-im

In [10]:
gt_SBB=pd.read_csv('../gt-SBB.tsv', sep='\t')
gt_NLF=pd.read_csv('../gt-NLF.tsv', sep='\t')
gt_BnF=pd.read_csv('../gt-BnF.tsv', sep='\t')

In [11]:
def evaluate_matching_result(gt_name, ocr_name):

    gt_tsv_file = f"../gt-{gt_name}.tsv"
    match_tsv_file = f"../match-gt-{gt_name}-{ocr_name if ocr_name != 'GT' else 'gt'}-layout-ocr.tsv"

    if not os.path.exists(gt_tsv_file) or not os.path.exists(match_tsv_file):
        return None

    gt = pd.read_csv(gt_tsv_file, sep='\t')

    gt_num_pages = len(gt[['zdb', 'year', 'month', 'day', 'issue', 'page']].drop_duplicates())

    df = pd.read_csv(match_tsv_file, sep='\t', low_memory=False)

    matched_total_num_lines = len(df)

    ro_len_per_file = df[['xml_file', 'reading_order']].drop_duplicates().xml_file.value_counts()

    files_without_reading_order = list(ro_len_per_file.loc[ro_len_per_file == 1].index)

    gt = gt.loc[~gt.xml_file.isin(files_without_reading_order)].copy().reset_index(drop=True)
    df = df.loc[~df.xml_file.isin(files_without_reading_order)].copy().reset_index(drop=True)

    matched_no_reading_order = (df.reading_order == -1)

    df = df.loc[~matched_no_reading_order].copy().reset_index()

    df['prev_sequence_id'] = df.shift(1).sequence_id

    df['next_sequence_id'] = df.shift(-1).sequence_id

    def compute_out_of_context(df_match):
        sequence_next_combis = pd.DataFrame([(sequence_id, next_sequence_id, len(tmp))
                                            for (sequence_id, next_sequence_id), tmp in
                                            df_match.groupby(['sequence_id', 'next_sequence_id'])],
                                            columns=["sid", "nid", "occ"])

        between_sequence_jumps = sequence_next_combis.loc[sequence_next_combis.sid != sequence_next_combis.nid]

        peseq = between_sequence_jumps.sid.value_counts()

        oocc =\
            pd.DataFrame(peseq.value_counts()).\
                rename(columns={"count": "#articles"}).\
                reset_index().\
                rename(columns={"count": "#context switches"})

        return oocc, peseq

    gt_art_pages = gt[['sequence_id', 'page']].drop_duplicates()

    matched_art_pages = df[['sequence_id', 'page']].drop_duplicates()

    gt_multi_part_articles_on_one_page = gt.loc[gt[['sequence_id', 'page']].duplicated()].sequence_id.unique()

    gt_num_multi_part_articles_on_one_page = len(gt_multi_part_articles_on_one_page)

    matched_out_of_context_changes, _ = compute_out_of_context(df)

    matched_multi_part_on_one_page_out_of_context_changes, per_sequence =\
        compute_out_of_context(df.loc[df.sequence_id.isin(gt_multi_part_articles_on_one_page)])

    gt_articles_over_multiple_pages =\
        pd.DataFrame(gt_art_pages.sequence_id.\
            value_counts().\
            value_counts()).\
            rename(columns={"count": "#articles"}).\
            reset_index().\
            rename(columns={"count": "#pages"})

    matched_articles_over_multiple_pages =\
        pd.DataFrame(matched_art_pages.sequence_id.\
            value_counts().\
            value_counts()).\
            rename(columns={"count": "#articles"}).\
            reset_index().\
            rename(columns={"count": "#pages"})

    matched_textline_intersection =\
        pd.DataFrame(df.num_matches.value_counts()).reset_index()

    gt_tag_distribution = pd.DataFrame(gt.tag.value_counts()).reset_index()

    return { 'gt_name' : gt_name, 
             'ocr_name': ocr_name, 
             'gt': gt,
             'matched' : df,
             'gt_num_pages': gt_num_pages, 
             'matched_total_num_lines': matched_total_num_lines,
             'files_without_reading_order': files_without_reading_order,
             'matched_no_reading_order': matched_no_reading_order,
             'gt_art_pages': gt_art_pages,
             'matched_art_pages': matched_art_pages,
             'gt_multi_part_articles_on_one_page': gt_multi_part_articles_on_one_page,
             'gt_num_multi_part_articles_on_one_page': gt_num_multi_part_articles_on_one_page,
             'matched_out_of_context_changes': matched_out_of_context_changes,
             'matched_multi_part_on_one_page_out_of_context_changes' : matched_multi_part_on_one_page_out_of_context_changes,
             'gt_articles_over_multiple_pages': gt_articles_over_multiple_pages, 
             'matched_articles_over_multiple_pages': matched_articles_over_multiple_pages, 
             'matched_textline_intersection': matched_textline_intersection,
             'gt_tag_distribution': gt_tag_distribution }

In [12]:
gt_names = ['SBB','NLF','BnF']
ocr_names = ['GT', 'eynollah','pero']

evaluation = dict()

gt_info = dict()

def get_configs():
    for gtn in gt_names:
        for on in ocr_names:
            yield gtn,on

for gtn,on in tqdm(get_configs(), total=len(gt_names)*len(ocr_names)):

    result = evaluate_matching_result(gtn, on)

    if result is None:
        continue

    evaluation[(gtn,on)] = result
    if gtn not in gt_info: 
        gt_info[gtn] = result # just for GT


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:42<00:00,  4.74s/it]


In [13]:
def compute_RAC(csw, total):

    rac = 0.0
    for num_sw, (count,) in csw.iterrows():
        rac += 1.0/float(num_sw) * float(count)

    rac /= float(total)
        
    return rac

def compute_out_of_context_table(res_key, total_key):
    moocc=None
    columns = [('','#context switches')]
    for gtn in ['NLF','BnF']:
    
        columns.append(("GT", gtn))
        tmp = evaluation[(gtn,'GT')][res_key].copy().rename(columns={"#articles": f"{gtn}_gt"})
        
        moocc = pd.DataFrame(tmp) if moocc is None else moocc.merge(tmp, on="#context switches", how="outer")
    
    moocc
    for on in ['eynollah','pero']:
        for gtn in gt_names:
           
            columns.append((on, gtn))
            tmp = evaluation[(gtn, on)][res_key].copy().rename(columns={"#articles": f"{gtn}_{on}"})
            moocc = moocc.merge(tmp, on="#context switches", how="outer")
        
    moocc.columns=pd.MultiIndex.from_tuples(columns)

    moocc[moocc.isnull()]=0.0
    moocc = moocc.astype(int)
    moocc[moocc=="0"]='-'
    moocc = moocc.set_index(('','#context switches'))
    moocc.index = moocc.index.rename('context switches')


    total_num = pd.DataFrame([], columns=moocc.columns)

    if total_key == 'gt':
        for c in moocc.columns:
            total_num.loc['total #articles', c] = len(evaluation[(c[1],c[0])]['gt'].sequence_id.unique())
    elif total_key == 'gt_num_multi_part_articles_on_one_page':
        for c in moocc.columns:
            total_num.loc['total #articles', c] = evaluation[(c[1],c[0])]['gt_num_multi_part_articles_on_one_page']
        
    
    RAC = pd.DataFrame([], columns=moocc.columns)
    
    for c in moocc.columns:
        RAC.loc['RAC', c] = round(compute_RAC(pd.DataFrame(moocc.loc[:, c]), total_num.loc['total #articles', c]),3)

    empty = pd.DataFrame([], columns=moocc.columns)
    empty.loc['context switches:',:] = ''

    moocc = moocc.iloc[0:9]
    
    moocc = pd.concat([total_num, RAC, empty, moocc])

    return moocc


In [14]:
moocc = compute_out_of_context_table('matched_out_of_context_changes', 'gt')


with open('matched_out_of_context_changes.tex','w') as f:
    f.write(moocc.to_latex(column_format="r@{\hskip 0.15in}rr@{\hskip 0.25in}rrr@{\hskip 0.25in}rrr", float_format="{:0.3f}".format).replace('_','\_'))

moocc

GT        eynollah                pero              
                     NLF    BnF      SBB    NLF   BnF    SBB    NLF    BnF
total #articles     6564   6640     7216   6564  6640   7119   6564   6640
RAC                0.997  0.974    0.916  0.949   0.9  0.754  0.602  0.543
context switches:                                                         
1                   6528   6319     6126   5966  5500   4768   3460   2838
2                     31    266      848    446   788   1001    814   1188
3                      4     41      134     75   157    208    176    366
4                      0      9       44     34    68     76     57    135
5                      0      3        9     15    25     22     26     48
6                      0      0        2      6    21     16      8     26
7                      0      1        1      8     8      4      8     14
8                      0      0        0      2    16      1      2      9
9                      0      0        0      3     8      1      3      3

In [15]:
moocc2 = compute_out_of_context_table('matched_multi_part_on_one_page_out_of_context_changes', 'gt_num_multi_part_articles_on_one_page')


with open('matched_multi_part_on_one_page_out_of_context_changes.tex','w') as f:
    f.write(moocc2.to_latex(column_format="r@{\hskip 0.15in}rr@{\hskip 0.25in}rrr@{\hskip 0.25in}rrr", float_format="{:0.3f}".format).replace('_','\_'))

moocc2

GT       eynollah                 pero              
                     NLF   BnF      SBB    NLF    BnF    SBB    NLF    BnF
total #articles     4292  5643      477   4292   5643    477   4292   5643
RAC                0.996  0.97    0.812  0.934  0.894  0.579  0.479  0.541
context switches:                                                         
1                   4257  5324      307   3791   4608    102   1598   2296
2                     31   265      154    369    726    313    766   1174
3                      4    41        7     65    138     46    169    358
4                      0     9        3     33     63      7     55    132
5                      0     3        2     14     19      2     19     44
6                      0     0        0      5     20      2      6     24
7                      0     1        0      6      8      0      7     11
8                      0     0        0      2     16      0      2      7
9                      0     0        0      2      8      0      2      3

In [16]:
def compute_text_lines_table(res_key):
    tli=None
    columns = [('','num_matches')]
    for gtn in ['NLF','BnF']:
    
        columns.append(("GT", gtn))
        tmp = evaluation[(gtn,'GT')][res_key].copy().rename(columns={"count": f"{gtn}_gt"})
        
        tli = pd.DataFrame(tmp) if tli is None else tli.merge(tmp, on="num_matches", how="outer")
    
    tli
    for on in ['eynollah','pero']:
        for gtn in gt_names:
            columns.append((on, gtn))
            tmp = evaluation[(gtn, on)][res_key].copy().rename(columns={"count": f"{gtn}_{on}"})
            tli = tli.merge(tmp, on="num_matches", how="outer")
        
    tli.columns=pd.MultiIndex.from_tuples(columns)

    tli = tli.set_index(('','num_matches'))

    total_num = pd.DataFrame([], columns=tli.columns)

    for c in tli.columns:
        total_num.loc['total textlines', c] = len(evaluation[(c[1],c[0])]['matched'])

    for i in tli.index:
        tli.loc[i] = (tli.loc[i].astype(float)/total_num.loc['total textlines'].astype(float)).round(2)

    empty = pd.DataFrame([], columns=tli.columns)
    empty.loc['num intersections:',:] = ''

    tli[tli.isnull()] = 0.0
    #tli = tli.astype(int)
    
    tli = pd.concat([total_num, empty, tli])

    return tli

In [17]:
text_line_matches = compute_text_lines_table('matched_textline_intersection')

with open('text_line_matches.tex','w') as f:
    f.write(text_line_matches.to_latex(column_format="r@{\hskip 0.15in}rr@{\hskip 0.25in}rrr@{\hskip 0.25in}rrr", float_format="{:0.2f}".format).replace('_','\_').replace('0.00','0.0'))

text_line_matches

GT         eynollah                    pero          \
                       NLF     BnF      SBB     NLF     BnF     SBB     NLF   
total textlines     132309  187704   174009  135721  161418  156459  115492   
num intersections:                                                            
0                      0.0     0.0     0.01    0.02    0.02    0.05    0.05   
1                     0.74    0.61     0.93    0.82    0.68    0.88     0.8   
2                     0.22     0.3     0.06    0.15    0.26    0.06    0.13   
3                     0.04    0.08      0.0    0.02    0.04     0.0    0.02   
4                      0.0    0.01      0.0     0.0     0.0     0.0     0.0   
5                      0.0     0.0      0.0     0.0     0.0     0.0     0.0   
6                      0.0     0.0      0.0     0.0     0.0     0.0     0.0   
7                      0.0     0.0      0.0     0.0     0.0     0.0     0.0   
8                      0.0     0.0      0.0     0.0     0.0     0.0     0.0   
9                      0.0     0.0      0.0     0.0     0.0     0.0     0.0   
10                     0.0     0.0      0.0     0.0     0.0     0.0     0.0   
15                     0.0     0.0      0.0     0.0     0.0     0.0     0.0   

                            
                       BnF  
total textlines     155683  
num intersections:          
0                     0.03  
1                     0.68  
2                     0.25  
3                     0.05  
4                      0.0  
5                      0.0  
6                      0.0  
7                      0.0  
8                      0.0  
9                      0.0  
10                     0.0  
15                     0.0

In [18]:
def compute_text_lines_table2():

    # columns = [('','num_matches')]
    columns =[]
    for gtn in ['NLF','BnF']:
        columns.append(("GT", gtn))
    
    for on in ['eynollah','pero']:
        for gtn in gt_names:
            columns.append((on, gtn))
        
    intersection = pd.DataFrame([], columns=pd.MultiIndex.from_tuples(columns))

    steps = [0.5, 0.6, 0.7, 0.8, 0.9, 0.91, 0.92, 0.93, 0.94, 0.95]
    for c in intersection.columns:
        #import ipdb;ipdb.set_trace()

        total = len(evaluation[(c[1],c[0])]['matched'])
        
        intersection.loc['total textlines', c] = total

        for s in steps:
            intersection.loc[f'intersection > {s}', c] = ((evaluation[(c[1],c[0])]['matched'].match_score > s).sum()/total).round(2)
    
    return intersection

In [19]:
textline_intersection = compute_text_lines_table2()

with open('textline_intersection.tex','w') as f:
    f.write(textline_intersection.to_latex(column_format="l@{\hskip 0.15in}rr@{\hskip 0.25in}rrr@{\hskip 0.25in}rrr", float_format="{:0.2f}".format).replace('_','\_').replace('0.00','0.0'))

textline_intersection

GT         eynollah                    pero          \
                        NLF     BnF      SBB     NLF     BnF     SBB     NLF   
total textlines      132309  187704   174009  135721  161418  156459  115492   
intersection > 0.5      1.0    0.98     0.99    0.98    0.97    0.95    0.95   
intersection > 0.6     0.99    0.96     0.99    0.98    0.96    0.94    0.95   
intersection > 0.7     0.98    0.92     0.99    0.98    0.95    0.94    0.94   
intersection > 0.8     0.95    0.82     0.98    0.97    0.92    0.94    0.93   
intersection > 0.9     0.86    0.67     0.97    0.93    0.82    0.93    0.87   
intersection > 0.91    0.85    0.65     0.97    0.92    0.81    0.93    0.86   
intersection > 0.92    0.83    0.63     0.97    0.91    0.79    0.93    0.85   
intersection > 0.93    0.82    0.61     0.96     0.9    0.78    0.93    0.84   
intersection > 0.94     0.8    0.59     0.96    0.89    0.76    0.92    0.83   
intersection > 0.95    0.78    0.57     0.96    0.88    0.74    0.92    0.82   

                             
                        BnF  
total textlines      155683  
intersection > 0.5     0.95  
intersection > 0.6     0.95  
intersection > 0.7     0.94  
intersection > 0.8     0.91  
intersection > 0.9     0.81  
intersection > 0.91    0.79  
intersection > 0.92    0.77  
intersection > 0.93    0.75  
intersection > 0.94    0.72  
intersection > 0.95     0.7

In [20]:
zdb_meta = get_zdb_meta_data(gt_SBB[['zdb']].rename(columns={'zdb':'zdb_id'}).drop_duplicates())

Retrieving ZDB meta data ...: 100%|███████████████████████████████████████████████████████████████████████| 48/48 [00:06<00:00,  7.12it/s]


In [21]:
zdb_info = zdb_meta.copy().drop(columns=['creator', 'publisher']).rename(columns={'date': 'publication period'})

for zdb, publication in gt_SBB.groupby('zdb'):

    pages = publication[['year','month','day','issue', 'page']].drop_duplicates()

    zdb_info.loc[zdb, 'num_pages'] = len(pages)

    min_year = pages.year.min()
    max_year = pages.year.max()

    if min_year == max_year:
        zdb_info.loc[zdb, 'sample period'] = str(min_year)
    else:
        zdb_info.loc[zdb, 'sample period'] = f"{min_year}-{max_year}"

# no ZDB data for ID 24344771 : Volks-Zeitung & 1853-1904  & 1858 & 3 & ger \\  
zdb_info.loc["24344771","title"] = "Volks-Zeitung"
zdb_info.loc["24344771","full_title"] = "Volks-Zeitung"
zdb_info.loc["24344771","publication period"] = "1853-1904"
zdb_info.loc["24344771","sample period"] = "1858"
zdb_info.loc["24344771","language"] = "ger"

zdb_info['full_title'] = zdb_info.title
zdb_info['title'] = zdb_info.title.str.extract("(.*?):.*")
zdb_info.loc[zdb_info.title.isnull(), 'title'] = zdb_info.loc[zdb_info.title.isnull(), 'full_title']
zdb_info['num_pages'] = zdb_info.num_pages.astype(int)
zdb_info = zdb_info[['title', 'publication period', 'sample period', 'num_pages', 'language']]
zdb_info

,title,publication period,sample period,num_pages,language
zdb_id,,,,,
11614109,Neueste Mittheilungen,1882-1894,1883-1894,9,ger
23820457,Deutsch-Ostafrikanische Zeitung,1899-1916,1904-1913,12,ger
2432291X,Bütower Anzeiger,1889-1918,1891,3,ger
24324218,Frankensteiner Kreisblatt,1877-1896,1878,3,ger
24329435,Deutsch-chinesische Nachrichten,1930-1939,1933,6,ger
24332471,Berliner Gerichts-Zeitung,1853-1898,1882-1885,6,ger
24336117,Münsterberger Kreisblatt,1888-1931,1930,3,ger
24340492,National-Zeitung,1848-1910,1848-1902,18,ger
24344771,Volks-Zeitung,1853-1904,1858,3,ger


In [22]:
zdb_info.num_pages.sum()

np.int64(500)

In [23]:
with open('zdb_info.tex','w') as f:
    f.write(zdb_info.to_latex(index=False, column_format="lrrrr").replace('_','\_'))

In [24]:
gt_tag_dist = None
for gtn in gt_names:
    td = gt_info[gtn]['gt_tag_distribution'].rename(columns={'count': gtn})

    gt_tag_dist = td if gt_tag_dist is None else gt_tag_dist.merge(td, on="tag", how="outer")

gt_tag_dist[gt_tag_dist.isnull()]=0.0
gt_tag_dist = gt_tag_dist.set_index('tag').astype(int).astype(str)
gt_tag_dist[gt_tag_dist=="0"]='-'
gt_tag_dist = gt_tag_dist.loc[gt_tag_dist.index!="not_specified"]

gt_tag_dist

,SBB,NLF,BnF
tag,,,
advertisement,932,-,-
article,4500,2308,1302
article_head,774,4299,5820
article_middle_part,10,16770,49619
article_tail,710,4299,5820
heading,692,-,-
obituary,17,-,-
page_footer,23,-,-
page_header,132,-,-


In [25]:
with open('gt_tag_distribution.tex','w') as f:
    f.write(gt_tag_dist.to_latex(column_format="lrrr").replace('_','\_'))

In [26]:
aomp = None
for gtn in gt_names:
    td = gt_info[gtn]['gt_articles_over_multiple_pages'].rename(columns={'#articles': gtn})

    aomp = td if aomp is None else aomp.merge(td, on="#pages", how="outer")

aomp[aomp.isnull()]=0.0
aomp = aomp.set_index('#pages').astype(int).astype(str)
# aomp[aomp=="0"]='-'

with open('count_multi_page_articles.tex','w') as f:
    f.write(aomp.to_latex(column_format="lrrrr").replace('_','\_'))

aomp


,SBB,NLF,BnF
#pages,,,
1,6996,6564,6640
2,217,0,0
3,3,0,0
